<a href="https://colab.research.google.com/github/Knguyen351/CIS-3400-Advanced-Programming-Kevin-Nguyen/blob/main/files_time_and_merges_Finished.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Working with Files, Time, and Merges
**Advanced Python with Dr. Thompson, Catawba College: tools for getting data in, lining it up, and trusting it**

## Table of Contents
- [1. Markdown and a Table of Contents](#1.-Markdown-and-a-Table-of-Contents)
- [2. Assert - Your Built-in Safety Net](#2.-Assert---Your-Built-in-Safety-Net)
- [3. Load a Local File with a Prompt](#3.-Load-a-Local-File-with-a-Prompt)
- [4. Connect Google Drive](#4.-Connect-Google-Drive)
- [5. Read Data Straight from GitHub](#5.-Read-Data-Straight-from-GitHub)
- [6. Merge Two Files on a Common Key](#6.-Merge-Two-Files-on-a-Common-Key)
- [7. Working with UTC](#7.-Working-with-UTC)
- [8. Merge Two Files on Timestamps](#8.-Merge-Two-Files-on-Timestamps)
- [9. Recap](#9.-Recap)

*(This list is plain Markdown. Section 1 shows how to write it and how to auto-generate one.)*

## Setup
Run this once. It creates three small CSV files we will use throughout (a bike-share scenario: rides, stations, and weather readings).


### UTC Time (Coordinated Universal Time)

1. **Definition:** UTC is the worldwide standard for measuring time.
2. **Purpose:** Provides a common time reference across countries and time zones.
3. **Time Zones:** Local times are expressed as offsets from UTC.
   - Eastern Standard Time (EST) = UTC−5
   - Eastern Daylight Time (EDT) = UTC−4
   - Pacific Standard Time (PST) = UTC−8
4. **Example:** If it is 12:00 PM UTC, it is 8:00 AM EDT.
5. **Computing:** Databases, APIs, cloud services, and applications commonly store timestamps in UTC to avoid time-zone confusion.
6. **Best Practice:** Store timestamps in UTC and convert them to local time for display.

**Python Example:**

```python
from datetime import datetime, timezone

now = datetime.now(timezone.utc)
print(now)
```

**Key Takeaway:** UTC provides one consistent clock worldwide, regardless of location or daylight saving time.


In [1]:
import io
import sqlite3
from datetime import datetime, timedelta, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd

print("pandas", pd.__version__)

# ---- build sample files (you don't need to read this cell) ----
rng = np.random.default_rng(7)
EASTERN = "America/New_York"

# Stations
stations = pd.DataFrame({
    "station_id": ["S01", "S02", "S03", "S04", "S05", "S06"],
    "station_name": ["Uptown", "Campus", "Library", "Stadium", "Depot", "Riverwalk"],
    "docks": [20, 15, 12, 30, 18, 10],
})
stations.to_csv("stations.csv", index=False)

# Rides: ~36 hours that straddle the end of daylight saving time (Sun Nov 1, 2026).
# Times are saved the way many systems save them: LOCAL wall-clock text with no offset.
start_utc = pd.Timestamp("2026-10-31 20:00", tz="UTC")           # 4:00 PM Eastern, Sat
ride_utc = start_utc + pd.to_timedelta(np.sort(rng.integers(0, 36 * 3600, 160)), unit="s")
ride_local_text = ride_utc.tz_convert(EASTERN).strftime("%Y-%m-%d %H:%M:%S")
rides = pd.DataFrame({
    "ride_id": [f"R{i:03d}" for i in range(160)],
    "station_id": rng.choice(["S01", "S02", "S03", "S04", "S05", "S06"], 160),
    "start_time": ride_local_text,
})
rides.loc[[10, 60, 120], "station_id"] = "S99"                    # three rides at an unknown station
rides.to_csv("rides.csv", index=False)

# Weather: a reading roughly every 15-20 minutes, stored in UTC with a trailing Z
w_utc = pd.Timestamp("2026-10-31 19:00", tz="UTC")
offsets = np.cumsum(rng.integers(900, 1200, 160))
w_times = w_utc + pd.to_timedelta(offsets, unit="s")
weather = pd.DataFrame({
    "timestamp": w_times.strftime("%Y-%m-%dT%H:%M:%SZ"),
    "temp_c": (12 + rng.normal(0, 1.5, 160)).round(1),
    "precip_mm": np.where(rng.random(160) < 0.15, rng.random(160).round(1), 0.0),
})
weather.to_csv("weather.csv", index=False)

for f in ["stations.csv", "rides.csv", "weather.csv"]:
    print(f, "->", len(pd.read_csv(f)), "rows")

pandas 2.2.3
stations.csv -> 6 rows
rides.csv -> 160 rows
weather.csv -> 160 rows


In [2]:
stations.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   station_id    6 non-null      object
 1   station_name  6 non-null      object
 2   docks         6 non-null      int64 
dtypes: int64(1), object(2)
memory usage: 276.0+ bytes


In [3]:
stations.head()

,station_id,station_name,docks
0,S01,Uptown,20
1,S02,Campus,15
2,S03,Library,12
3,S04,Stadium,30
4,S05,Depot,18


In [4]:
rides.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 160 entries, 0 to 159
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   ride_id     160 non-null    object
 1   station_id  160 non-null    object
 2   start_time  160 non-null    object
dtypes: object(3)
memory usage: 3.9+ KB


In [5]:
rides.head()

,ride_id,station_id,start_time
0,R000,S04,2026-10-31 16:08:03
1,R001,S02,2026-10-31 16:11:22
2,R002,S05,2026-10-31 16:16:23
3,R003,S06,2026-10-31 16:25:28
4,R004,S02,2026-10-31 16:30:48


In [6]:
weather.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 160 entries, 0 to 159
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   timestamp  160 non-null    object 
 1   temp_c     160 non-null    float64
 2   precip_mm  160 non-null    float64
dtypes: float64(2), object(1)
memory usage: 3.9+ KB


In [7]:
weather.head()

,timestamp,temp_c,precip_mm
0,2026-10-31T19:17:20Z,11.9,0.3
1,2026-10-31T19:33:23Z,12.0,0.0
2,2026-10-31T19:49:52Z,9.8,0.0
3,2026-10-31T20:09:26Z,11.3,0.0
4,2026-10-31T20:28:30Z,13.1,0.0


---
## 1. Markdown and a Table of Contents
*~4 min*

A notebook with eight sections is hard to navigate unless it has structure. Markdown gives you that for free.

**Headings create the outline.** Use `#` for the title, `##` for sections, `###` for subsections. Both Jupyter and VS Code build a clickable outline from them:
- **Jupyter / JupyterLab:** table of contents panel in the left sidebar
- **VS Code:** the **Outline** view (Explorer sidebar), or **Ctrl/Cmd+Shift+O** to jump to a heading

**Links to headings** are what make a TOC inside the notebook clickable:

```markdown
## Table of Contents
- [Setup](#Setup)
- [1. Loading Data](#1.-Loading-Data)
  - [Reading CSVs](#Reading-CSVs)
```
*(Jupyter style: the link is `#` + the heading text with spaces replaced by `-`, and case is kept.
VS Code and GitHub style: all lowercase, punctuation removed, so `#1-loading-data`.)*

**Other Markdown worth knowing:**

| You type | You get |
|---|---|
| `**bold**`, `*italic*`, `` `code` `` | **bold**, *italic*, `code` |
| `- item` / `1. item` | bullet / numbered list |
| `[text](https://...)` | link |
| `> note` | block quote (good for callouts) |
| `---` | horizontal rule (good between sections) |
| `$a^2 + b^2 = c^2$` | inline math |
| `<details><summary>Hint</summary>text</details>` | collapsible hint (works in Jupyter) |


## Using the Table of Contents in Google Colab

Google Colab automatically creates a **Table of Contents (TOC)** based on the Markdown headings in your notebook. No Python code is required!

### Step 1: Open the Table of Contents

1. Open your notebook in Google Colab.
2. Click the **Table of contents icon** on the left sidebar.
3. Colab automatically displays a list of headings from your notebook.
4. Click any heading to navigate directly to that section.

### Step 2: Create Markdown Headings

To organize your notebook, add a **Text cell** and use Markdown headings:

- `#` Main title (Heading 1)
- `##` Major section (Heading 2)
- `###` Subsection (Heading 3)

For example:

```text
# Working with Files and Time
## 1. Introduction
## 2. Reading CSV Files
### 2.1 Loading Data
## 3. Working with UTC
## 4. Merging DataFrames
### 4.1 Inner Joins
### 4.2 Left Joins
```

### Step 3: Navigate Your Notebook

- Colab automatically updates the TOC when you add or change headings.
- Click a heading in the TOC to jump to that section.
- Use meaningful headings to make your notebook easy to navigate.

### Optional: Add a Clickable TOC Inside Your Notebook

You can also create a Markdown cell at the beginning of your notebook containing links to sections.

To obtain a working link:

1. Find the heading cell you want to link to.
2. Click the three-dot menu on that cell.
3. Select **Copy link to cell**.
4. Paste the link into your TOC using Markdown link syntax:

`[Section Name](paste-cell-link-here)`

### Key Takeaway

**Use Markdown headings to organize your notebook.** Google Colab automatically builds a Table of Contents, making large notebooks easier to navigate without additional Python code.


### Auto-generate the TOC
If you don't want to maintain the links by hand, this "helper" may work - it does need adjustment for Colab or the environment you are using. This helper reads your `.ipynb`, finds the headings in Markdown cells (ignoring `#` comments inside code fences), and prints a TOC you can paste into the top cell.

In [8]:
import json
import re
from IPython.display import Markdown, display

def make_toc(notebook_path, style="jupyter", max_level=3):
    """Build a Markdown table of contents from a notebook's headings.
    style="jupyter" -> #Heading-Text      style="vscode" (or "github") -> #heading-text
    """
    nb = json.loads(Path(notebook_path).read_text(encoding="utf-8"))
    lines = []
    for cell in nb["cells"]:
        if cell["cell_type"] != "markdown":
            continue
        in_fence = False
        for line in "".join(cell["source"]).splitlines():
            if line.strip().startswith("```"):
                in_fence = not in_fence
            m = re.match(r"^(#{1,6})\s+(.*?)\s*$", line)
            if in_fence or not m:
                continue
            level, title = len(m.group(1)), m.group(2)
            if level == 1 or level > max_level:      # skip the notebook title
                continue
            if style == "jupyter":
                anchor = title.replace(" ", "-")
            else:
                anchor = re.sub(r"[^\w\- ]", "", title.lower()).replace(" ", "-")
            lines.append("  " * (level - 2) + f"- [{title}](#{anchor})")
    return "\n".join(lines)

# Change this to the name of THIS notebook file:
NOTEBOOK = "files_time_merges.ipynb"

if Path(NOTEBOOK).exists():
    toc_text = make_toc(NOTEBOOK, style="jupyter")   # use style="vscode" in VS Code
    print(toc_text)                                  # copy/paste this into your TOC cell
    display(Markdown(toc_text))                      # ...or preview it right here
else:
    print(f"Save this notebook as {NOTEBOOK!r} in the working folder (or edit NOTEBOOK) to try the TOC generator.")

Save this notebook as 'files_time_merges.ipynb' in the working folder (or edit NOTEBOOK) to try the TOC generator.


---
## 2. Assert - Your Built-in Safety Net
*~4 min*

`assert condition, "message"` stops the program the moment something you *believed* is false. Use it to document and enforce assumptions, especially before and after merges and time conversions (which we're about to do).

In [9]:
def mean_temp(df):
    """Average temperature of a weather DataFrame."""
    assert isinstance(df, pd.DataFrame), f"expected a DataFrame, got {type(df).__name__}"   # precondition
    assert "temp_c" in df.columns, f"temp_c missing; columns are {list(df.columns)}"
    result = df["temp_c"].mean()
    assert -60 < result < 60, f"implausible average temperature: {result}"                  # postcondition
    return result

print(mean_temp(pd.read_csv("weather.csv")))

11.936875


In [10]:
# What failure looks like
try:
    mean_temp(pd.read_csv("rides.csv"))
except AssertionError as e:
    print("AssertionError:", e)

AssertionError: temp_c missing; columns are ['ride_id', 'station_id', 'start_time']


### Three things worth knowing
1. **`assert` is for bugs in *your* code, not for bad user input.** Python's `-O` flag removes all asserts, so never rely on them to validate input. For user/file problems, `raise ValueError(...)` instead.
2. **Always write a message.** `assert x` tells you nothing at 2 AM; `assert x, "why it matters"` tells you everything.
3. **The tuple trap.** Parentheses around both parts make a non-empty tuple, which is always truthy, so the check silently never runs:

In [11]:
x = -5
assert (x > 0, "x must be positive")      # passes!  (Python prints a SyntaxWarning: it's a tuple)
print("That 'passed' even though x is", x)

That 'passed' even though x is -5


<>:2: SyntaxWarning: assertion is always true, perhaps remove parentheses?
<>:2: SyntaxWarning: assertion is always true, perhaps remove parentheses?
/tmp/ipykernel_1360/2398290842.py:2: SyntaxWarning: assertion is always true, perhaps remove parentheses?
  assert (x > 0, "x must be positive")      # passes!  (Python prints a SyntaxWarning: it's a tuple)


In [12]:
try:
    assert x > 0, "x must be positive"   # correct form
except AssertionError as e:
    print("Correct form caught it:", e)

Correct form caught it: x must be positive


---
## 3. Load a Local File with a Prompt
*~4 min*

Hard-coding `C:\\Users\\me\\Downloads\\data.csv` makes your notebook break on every other computer. Ask for the path instead and validate it.

**Option A: a text prompt with `input()`** works everywhere. Press **Enter** to use the sample file so the notebook keeps running.

In [13]:
def ask_for_file(prompt="Path to a CSV file (press Enter for the sample): ",
                 default="rides.csv", suffixes=(".csv", ".txt")):
    """Keep asking until the user gives a path that exists and has an allowed extension."""
    while True:
        text = input(prompt).strip().strip("\"'")        # drag-and-drop often adds quotes
        path = Path(text).expanduser() if text else Path(default)
        if not path.is_file():
            print(f"  Not found: {path}")
        elif path.suffix.lower() not in suffixes:
            print(f"  Expected one of {suffixes}, got {path.suffix!r}")
        else:
            return path

csv_path = ask_for_file()
rides_raw = pd.read_csv(csv_path)
print(f"Loaded {csv_path} -> {rides_raw.shape[0]} rows, {rides_raw.shape[1]} columns")
rides_raw.head()

Path to a CSV file (press Enter for the sample): weather.csv
Loaded weather.csv -> 160 rows, 3 columns


,timestamp,temp_c,precip_mm
0,2026-10-31T19:17:20Z,11.9,0.3
1,2026-10-31T19:33:23Z,12.0,0.0
2,2026-10-31T19:49:52Z,9.8,0.0
3,2026-10-31T20:09:26Z,11.3,0.0
4,2026-10-31T20:28:30Z,13.1,0.0


### Download rides.csv first from session storage! (Use left hand menu . . . files)

In [14]:
import pandas as pd
from google.colab import files
from io import BytesIO

uploaded = files.upload()

filename = next(iter(uploaded))

rides_raw = pd.read_csv(BytesIO(uploaded[filename]))

print(f"Loaded {filename} -> {rides_raw.shape[0]} rows, {rides_raw.shape[1]} columns")

display(rides_raw.head())

Saving rides.csv to rides (1).csv
Loaded rides (1).csv -> 160 rows, 3 columns


,ride_id,station_id,start_time
0,R000,S04,2026-10-31 16:08:03
1,R001,S02,2026-10-31 16:11:22
2,R002,S05,2026-10-31 16:16:23
3,R003,S06,2026-10-31 16:25:28
4,R004,S02,2026-10-31 16:30:48


---
## 4. Connect Google Drive
*~3 min*

How you connect depends on **where the notebook is running**:

| Where you run | How to reach Drive |
|---|---|
| **Google Colab** | `drive.mount()`: Google asks for permission, then Drive looks like a folder |
| **Your laptop** | Install **Google Drive for desktop**; your Drive appears as a normal folder/drive letter |
| **Anywhere, for a single shared file** | Use a share link converted to a download link (below) |

In [15]:
# Colab: this cell does nothing harmful elsewhere, it just explains itself.
try:
    from google.colab import drive
    drive.mount("/content/drive")               # pops up a permission prompt
    DRIVE_ROOT = Path("/content/drive/MyDrive")
    print("Drive mounted. Top-level items:", [p.name for p in DRIVE_ROOT.iterdir()][:10])
except ImportError:
    DRIVE_ROOT = None
    print("Not running in Colab. Use Drive for desktop (a normal folder path) or a share link.")

Mounted at /content/drive
Drive mounted. Top-level items: ['Cert 1.gdoc', 'RCCC Computer Science.gdoc', 'Untitled form.gform', 'Soccer Questions.gform', 'Career meet up UNCC.gdoc', 'Untitled document (38).gdoc', 'Untitled document (37).gdoc', 'The Walmart Trinity Code.gdoc', 'Math Learning Platform.gdoc', 'Activity 1.4 Chapter 1.gdoc']


**Using it** is just a path, the same as any local file:
```python
df = pd.read_csv(DRIVE_ROOT / "CIS3400" / "rides.csv")          # read
df.to_csv(DRIVE_ROOT / "CIS3400" / "rides_clean.csv", index=False)  # write back
```

**Drive for desktop paths** (replace with yours):
- Windows: `G:/My Drive/CIS3400/rides.csv`
- macOS: `~/Library/CloudStorage/GoogleDrive-you@school.edu/My Drive/CIS3400/rides.csv`

**Share links (no login code).** Set the file to *"Anyone with the link can view"*, copy the file ID from the URL (`.../d/<FILE_ID>/view`), then:
```python
FILE_ID = "paste-id-here"
df = pd.read_csv(f"https://drive.google.com/uc?export=download&id={FILE_ID}")
```
For a **Google Sheet**, use `https://docs.google.com/spreadsheets/d/<SHEET_ID>/export?format=csv`.
Share links only work for public/link-shared files, and large files can hit a Google confirmation page, so for anything big or private, use `drive.mount` or Drive for desktop.

---
## 5. Read Data Straight from GitHub
*~3 min*

**Yes, you can read data straight from GitHub.** `pd.read_csv` accepts a URL. The catch: the normal GitHub page URL returns an HTML web page. You need the **raw** URL.

| | URL |
|---|---|
| Page (HTML, won't work) | `https://github.com/mwaskom/seaborn-data/blob/master/tips.csv` |
| **Raw (works)** | `https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv` |

On GitHub you can also click the **Raw** button on any file and copy the address. Or convert it in code:

In [16]:
def github_raw(url):
    """Convert a github.com .../blob/... link into its raw.githubusercontent.com link."""
    if url.startswith("https://github.com/") and "/blob/" in url:
        url = url.replace("https://github.com/", "https://raw.githubusercontent.com/", 1).replace("/blob/", "/", 1)
    return url

page_url = "https://github.com/mwaskom/seaborn-data/blob/master/tips.csv"
raw_url = github_raw(page_url)
assert raw_url == "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv", raw_url
print(raw_url)

try:
    tips = pd.read_csv(raw_url)          # needs an internet connection
    print(f"Loaded {len(tips)} rows")
    display(tips.head())
except Exception as e:                   # offline, firewall, typo in URL...
    tips = None
    print(f"Could not download ({type(e).__name__}). Check your connection and the URL.")

https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv
Loaded 244 rows


,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


**Good practice**
- **Pin a version.** `master` can change under you. Replace the branch name with a **commit hash** or tag in the URL for reproducible results (`.../seaborn-data/<commit-sha>/tips.csv`).
- **Raw files are cached** for a few minutes, so a fresh push may not show up immediately.
- **Private repo?** The raw URL won't work without credentials. Use a personal access token from an environment variable (never paste tokens into a notebook you'll share):
```python
import os, requests
token = os.environ["GITHUB_TOKEN"]
r = requests.get(
    "https://api.github.com/repos/OWNER/REPO/contents/data/file.csv",
    headers={"Authorization": f"Bearer {token}", "Accept": "application/vnd.github.raw+json"},
)
r.raise_for_status()
df = pd.read_csv(io.StringIO(r.text))
```
- **Need the whole repo?** `git clone` (or `git pull` to update) instead of downloading file by file.

---
## 6. Merge Two Files on a Common Key
*~4 min*

`pd.merge` is a SQL join. Decide three things: **which column matches** (`on`), **which rows to keep** (`how`), and **what relationship you expect** (`validate`).

| `how=` | Keeps |
|---|---|
| `"inner"` | only rows that match in both |
| `"left"` | every row from the left, matches from the right when they exist |
| `"right"` / `"outer"` | mirror image / everything from both |

In [17]:
rides.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 160 entries, 0 to 159
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   ride_id     160 non-null    object
 1   station_id  160 non-null    object
 2   start_time  160 non-null    object
dtypes: object(3)
memory usage: 3.9+ KB


In [18]:
stations.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   station_id    6 non-null      object
 1   station_name  6 non-null      object
 2   docks         6 non-null      int64 
dtypes: int64(1), object(2)
memory usage: 276.0+ bytes


In [19]:
rides = pd.read_csv("rides.csv")
stations = pd.read_csv("stations.csv")

merged = rides.merge(
    stations,
    on="station_id",             # use left_on="a", right_on="b" when the names differ
    how="left",                  # keep every ride, even if the station is unknown
    validate="m:1",              # many rides : one station. Raises if stations has duplicates
    indicator=True,              # adds a _merge column: both / left_only / right_only
)

merged["_merge"].value_counts()

,count
_merge,
both,157
left_only,3
right_only,0


In [20]:
merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 160 entries, 0 to 159
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   ride_id       160 non-null    object  
 1   station_id    160 non-null    object  
 2   start_time    160 non-null    object  
 3   station_name  157 non-null    object  
 4   docks         157 non-null    float64 
 5   _merge        160 non-null    category
dtypes: category(1), float64(1), object(4)
memory usage: 6.7+ KB


In [21]:
merged.head()

,ride_id,station_id,start_time,station_name,docks,_merge
0,R000,S04,2026-10-31 16:08:03,Stadium,30.0,both
1,R001,S02,2026-10-31 16:11:22,Campus,15.0,both
2,R002,S05,2026-10-31 16:16:23,Depot,18.0,both
3,R003,S06,2026-10-31 16:25:28,Riverwalk,10.0,both
4,R004,S02,2026-10-31 16:30:48,Campus,15.0,both


In [22]:
orphans = merged[merged["_merge"] == "left_only"]
orphans[["ride_id", "station_id", "station_name"]]

,ride_id,station_id,station_name
10,R010,S99,NaN
60,R060,S99,NaN
120,R120,S99,NaN


`indicator=True` just caught three rides at station `S99`, which doesn't exist in the stations file. Decide what to do (fix, drop, or investigate), then **lock in the expectation with asserts**:

In [23]:
rides_ok = merged[merged["_merge"] == "both"].drop(columns="_merge")

assert len(merged) == len(rides), "a left merge on a unique key must not change the row count"
assert rides_ok["station_name"].notna().all()
print(f"Kept {len(rides_ok)} of {len(rides)} rides; dropped {len(orphans)} with unknown stations")

Kept 157 of 160 rides; dropped 3 with unknown stations


**The classic merge bug: duplicate keys multiply rows.** `validate=` turns a silent problem into a loud one.

In [24]:
stations_dup = pd.concat([stations, stations.iloc[[0]]])      # S01 accidentally appears twice

silent = rides.merge(stations_dup, on="station_id", how="left")
print(f"Without validate: {len(rides)} rides became {len(silent)} rows (S01 rides doubled)")

try:
    rides.merge(stations_dup, on="station_id", how="left", validate="m:1")
except pd.errors.MergeError as e:
    print("With validate='m:1':", e)

Without validate: 160 rides became 184 rows (S01 rides doubled)
With validate='m:1': Merge keys are not unique in right dataset; not a many-to-one merge


---
## 7. Working with UTC
*~6 min*

**The rule:** store, compare, and merge in **UTC**. Convert to local time only for display.

UTC never has daylight saving, so one UTC moment is always exactly one moment. Local time can skip an hour (spring) or **repeat** one (fall). This Sunday, **Nov 1, 2026**, clocks in the US Eastern zone go back at 2:00 AM, so 1:30 AM happens *twice*.

**Naive vs aware:** a *naive* timestamp has no timezone (`2026-11-01 01:30`). An *aware* one carries it (`2026-11-01 01:30-04:00`). Mixing them is how time bugs happen.

### Pure Python

In [25]:
now_utc = datetime.now(timezone.utc)                    # aware, in UTC  (preferred)
print("UTC now:     ", now_utc.isoformat())
print("Eastern now: ", now_utc.astimezone(ZoneInfo("America/New_York")).isoformat())
print("Tokyo now:   ", now_utc.astimezone(ZoneInfo("Asia/Tokyo")).isoformat())

# Avoid these: datetime.utcnow() is deprecated (returns a NAIVE value), and datetime.now() is naive local time.

UTC now:      2026-10-28T16:30:29.529493+00:00
Eastern now:  2026-10-28T12:30:29.529493-04:00
Tokyo now:    2026-10-29T01:30:29.529493+09:00


### pandas: parse, localize, convert

In [26]:
# 1) Strings that already say UTC (the trailing Z) -> parse as UTC
weather = pd.read_csv("weather.csv")
weather["timestamp"] = pd.to_datetime(weather["timestamp"], utc=True)
print(weather["timestamp"].dtype)               # datetime64[..., UTC]  -> aware
weather.head(3)

datetime64[ns, UTC]


,timestamp,temp_c,precip_mm
0,2026-10-31 19:17:20+00:00,11.9,0.3
1,2026-10-31 19:33:23+00:00,12.0,0.0
2,2026-10-31 19:49:52+00:00,9.8,0.0


In [27]:
# 2) Local text with NO zone information -> you must say which zone it was recorded in
rides = pd.read_csv("rides.csv")
rides["start_local"] = pd.to_datetime(rides["start_time"])      # naive
print(rides["start_local"].dtype)                               # datetime64[...] -> NAIVE

try:
    rides["start_local"].dt.tz_localize("America/New_York")
except Exception as e:
    print(type(e).__name__, ":", str(e)[:120])

datetime64[ns]
AmbiguousTimeError : Cannot infer dst time from 2026-11-01 01:10:31, try using the 'ambiguous' argument


That error is the DST fall-back hour showing up in our data: some rides were logged at e.g. `01:30` and there are *two* real 1:30s that night. Here is the same problem on one timestamp:

In [28]:
ts = pd.Timestamp("2026-11-01 01:30")

first  = ts.tz_localize("America/New_York", ambiguous=True)     # True  = daylight time (EDT, first 1:30)
second = ts.tz_localize("America/New_York", ambiguous=False)    # False = standard time (EST, second 1:30)
print(first,  "->", first.tz_convert("UTC"))
print(second, "->", second.tz_convert("UTC"))
print("They are", second - first, "apart")

2026-11-01 01:30:00-04:00 -> 2026-11-01 05:30:00+00:00
2026-11-01 01:30:00-05:00 -> 2026-11-01 06:30:00+00:00
They are 0 days 01:00:00 apart


For a whole column that is **sorted in time order**, `ambiguous="infer"` figures out which 1:xx rides came first:

In [29]:
rides["start_utc"] = (
    rides["start_local"]
    .dt.tz_localize("America/New_York", ambiguous="infer")
    .dt.tz_convert("UTC")
)

assert rides["start_utc"].dt.tz is not None, "start_utc must be timezone-aware"
assert rides["start_utc"].is_monotonic_increasing, "UTC times should be in order (confirms the 'infer' guess)"

rides[["ride_id", "start_time", "start_utc"]].iloc[[0, 1, 2, -2, -1]]

,ride_id,start_time,start_utc
0,R000,2026-10-31 16:08:03,2026-10-31 20:08:03+00:00
1,R001,2026-10-31 16:11:22,2026-10-31 20:11:22+00:00
2,R002,2026-10-31 16:16:23,2026-10-31 20:16:23+00:00
158,R158,2026-11-02 02:45:25,2026-11-02 07:45:25+00:00
159,R159,2026-11-02 02:50:16,2026-11-02 07:50:16+00:00


In [30]:
# Look at the repeated hour: same local text, different UTC moments
repeat = rides[rides["start_time"].str.startswith("2026-11-01 01")]
repeat[["ride_id", "start_time", "start_utc"]]

,ride_id,start_time,start_utc
36,R036,2026-11-01 01:10:31,2026-11-01 05:10:31+00:00
37,R037,2026-11-01 01:35:08,2026-11-01 05:35:08+00:00
38,R038,2026-11-01 01:38:00,2026-11-01 05:38:00+00:00
39,R039,2026-11-01 01:01:23,2026-11-01 06:01:23+00:00
40,R040,2026-11-01 01:06:38,2026-11-01 06:06:38+00:00
41,R041,2026-11-01 01:15:44,2026-11-01 06:15:44+00:00
42,R042,2026-11-01 01:48:21,2026-11-01 06:48:21+00:00
43,R043,2026-11-01 01:48:54,2026-11-01 06:48:54+00:00
44,R044,2026-11-01 01:54:33,2026-11-01 06:54:33+00:00


### Why it matters: elapsed time
From 12:30 AM to 3:30 AM on Nov 1 looks like 3 hours on the wall clock. It is really **4**.

In [31]:
a = pd.Timestamp("2026-11-01 00:30")
b = pd.Timestamp("2026-11-01 03:30")
print("Naive difference:  ", b - a)

a_utc = a.tz_localize("America/New_York").tz_convert("UTC")
b_utc = b.tz_localize("America/New_York").tz_convert("UTC")
print("Aware (UTC) difference:", b_utc - a_utc)

Naive difference:   0 days 03:00:00
Aware (UTC) difference: 0 days 04:00:00


### Cheat sheet
| Task | Code |
|---|---|
| Parse strings that include an offset or `Z` | `pd.to_datetime(s, utc=True)` |
| Naive local -> aware | `s.dt.tz_localize("America/New_York")` |
| Aware -> another zone | `s.dt.tz_convert("UTC")` |
| Epoch seconds -> UTC | `pd.to_datetime(s, unit="s", utc=True)` |
| Current time, aware | `datetime.now(timezone.utc)` |
| Show in local time (display only) | `s.dt.tz_convert("America/New_York")` |
| Ambiguous / nonexistent local times | `ambiguous="infer"` / `True` / `False`, `nonexistent="shift_forward"` |

Use IANA names like `"America/New_York"`, **not** `"EST"`: the name knows about daylight saving, the abbreviation doesn't.

---
## 8. Merge Two Files on Timestamps
*~5 min*

Goal: attach the weather at the time of each ride. Rides happen at 4:07:33 PM; weather readings come at 4:00:12 and 4:17:45. Timestamps almost never match *exactly*, so an ordinary merge fails.

In [32]:
weather = weather.rename(columns={"timestamp": "weather_utc"})

exact = rides.merge(weather, left_on="start_utc", right_on="weather_utc", how="inner")
print(f"Exact-match merge found {len(exact)} matches out of {len(rides)} rides")

Exact-match merge found 0 matches out of 160 rides


**`pd.merge_asof`** matches each row to the **nearest** timestamp instead. Rules (all easy to check with `assert`):
1. **Both** tables sorted by their time column
2. **Same timezone awareness** (both aware, ideally both UTC)
3. Use `direction`: `"backward"` (latest reading *before* the ride), `"forward"`, or `"nearest"`
4. Use `tolerance` so an hours-old reading doesn't get matched by accident

In [33]:
rides_sorted = rides.sort_values("start_utc")
weather_sorted = weather.sort_values("weather_utc")

assert rides_sorted["start_utc"].is_monotonic_increasing
assert weather_sorted["weather_utc"].is_monotonic_increasing
assert rides_sorted["start_utc"].dt.tz == weather_sorted["weather_utc"].dt.tz, "timezone mismatch"

rides_weather = pd.merge_asof(
    rides_sorted,
    weather_sorted,
    left_on="start_utc",
    right_on="weather_utc",
    direction="backward",                  # most recent reading at or before the ride
    tolerance=pd.Timedelta("30min"),       # ...but not older than 30 minutes
)

rides_weather["reading_age"] = rides_weather["start_utc"] - rides_weather["weather_utc"]

assert len(rides_weather) == len(rides), "merge_asof should keep every ride"
assert (rides_weather["reading_age"].dropna() <= pd.Timedelta("30min")).all()

rides_weather[["ride_id", "start_utc", "weather_utc", "reading_age", "temp_c", "precip_mm"]].head()

,ride_id,start_utc,weather_utc,reading_age,temp_c,precip_mm
0,R000,2026-10-31 20:08:03+00:00,2026-10-31 19:49:52+00:00,0 days 00:18:11,9.8,0.0
1,R001,2026-10-31 20:11:22+00:00,2026-10-31 20:09:26+00:00,0 days 00:01:56,11.3,0.0
2,R002,2026-10-31 20:16:23+00:00,2026-10-31 20:09:26+00:00,0 days 00:06:57,11.3,0.0
3,R003,2026-10-31 20:25:28+00:00,2026-10-31 20:09:26+00:00,0 days 00:16:02,11.3,0.0
4,R004,2026-10-31 20:30:48+00:00,2026-10-31 20:28:30+00:00,0 days 00:02:18,13.1,0.0


In [34]:
print("Rides with no reading inside the tolerance:", rides_weather["temp_c"].isna().sum())
print("Median age of the matched reading:", rides_weather["reading_age"].median())

Rides with no reading inside the tolerance: 0
Median age of the matched reading: 0 days 00:06:54.500000


**What happens if you forget the timezone step?** Naive and aware timestamps refuse to merge, which is a *good* failure:

In [35]:
try:
    pd.merge_asof(
        rides_sorted.assign(start_utc=rides_sorted["start_local"]),   # naive local time
        weather_sorted,
        left_on="start_utc", right_on="weather_utc",
    )
except Exception as e:
    print(type(e).__name__, ":", str(e)[:160])

MergeError : incompatible merge keys [0] dtype('<M8[ns]') and datetime64[ns, UTC], must be the same type


**Simpler alternative: round both sides to a common bucket, then do a normal merge.** Less precise, but easy to explain, and good for "hourly weather":

In [36]:
weather_hourly = (
    weather.assign(hour=weather["weather_utc"].dt.floor("h"))
           .groupby("hour", as_index=False)
           .agg(avg_temp_c=("temp_c", "mean"), total_precip_mm=("precip_mm", "sum"))
)

rides_hourly = rides.assign(hour=rides["start_utc"].dt.floor("h")).merge(
    weather_hourly, on="hour", how="left", validate="m:1")

assert len(rides_hourly) == len(rides)
rides_hourly[["ride_id", "start_utc", "hour", "avg_temp_c", "total_precip_mm"]].head()

,ride_id,start_utc,hour,avg_temp_c,total_precip_mm
0,R000,2026-10-31 20:08:03+00:00,2026-10-31 20:00:00+00:00,12.1,0.0
1,R001,2026-10-31 20:11:22+00:00,2026-10-31 20:00:00+00:00,12.1,0.0
2,R002,2026-10-31 20:16:23+00:00,2026-10-31 20:00:00+00:00,12.1,0.0
3,R003,2026-10-31 20:25:28+00:00,2026-10-31 20:00:00+00:00,12.1,0.0
4,R004,2026-10-31 20:30:48+00:00,2026-10-31 20:00:00+00:00,12.1,0.0


**Also in the `merge_asof` toolbox:** `by="station_id"` (only match within the same station/sensor/ticker), `direction="nearest"`, and `allow_exact_matches=False`.

---
## 9. Recap

1. **Markdown:** headings + a TOC make long notebooks navigable. Generate it, don't hand-edit it.
2. **`assert`:** for your own assumptions (with a message); `raise` for bad input; never wrap a tuple.
3. **Local files:** prompt for the path, validate it, and don't hard-code `C:\\Users\\...`.
4. **Google Drive:** `drive.mount` in Colab, Drive for desktop on a laptop, share links for single public files.
5. **GitHub:** yes, use the **raw** URL (`github_raw()`), pin a commit for reproducibility.
6. **Merges:** choose `how`, use `validate=` and `indicator=True`, assert on row counts afterward.
7. **UTC:** store and compare in UTC, convert only for display, never mix naive and aware.
8. **Timestamps:** `merge_asof` (sorted, same tz, `tolerance`) or round to a bucket and merge normally.

### Now: project work time
Pick **one** idea and apply it to your project: add three `assert`s, switch a hard-coded path to a prompt, or merge in a second data source.